In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


Calculate rankings separately for RMSE, MASE, CRPS, QSS. Within every dataset, rank the models. 

Then calculate:

- the ordinary overall mean rank across all datasets
- leave-one-asset-class-out mean ranks, eight times
- equal-asset-class-weighted mean ranks
- the top model for each omission
- Spearman correlation between each leave-one-class-out ranking and the full ranking, which gives a compact numerical measure of ranking stability.

Lower is better for RMSE, MASE, CRPS, higher is better for QSS.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised\output_daily"
)

FILES = {
    "RMSE": ROOT / "rmse_summary_all_models.csv",
    "qCRPS": ROOT / "all_models_crps_by_dataset.csv",
    "QSS": ROOT / "qss_summary_all_models.csv",
    "MASE": ROOT / "mase_conventional_summary_all_models.csv",
}

OUT_DIR = ROOT / "robustness_rankings"
OUT_DIR.mkdir(exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def clean_dataset_name(x):
    """
    Standardise dataset names across the metric summary files.

    Examples:
      AUD_USD Historical Data_no_preprocess_lookback-250_horizon-20_slide-1
        -> AUD_USD Historical Data

      AUD_USD Historical Data
        -> AUD_USD Historical Data
    """
    x = str(x).strip()

    suffixes = [
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "_lookback-250_horizon-20_slide-1",
        "_no_preprocess",
    ]

    for suffix in suffixes:
        if x.endswith(suffix):
            x = x[:-len(suffix)]

    return x.strip()


def clean_model_name(x):
    """
    Minimal standardisation only.
    Add aliases here ONLY if inspection shows that the same model
    is written differently across files.
    """
    x = str(x).strip()

    aliases = {
        # Examples -- uncomment/change only if actually needed:
        # "GARCH": "GARCH(1,1)",
        # "Chronos": "Chronos-T5",
        # "Moirai MoE": "Moirai-MoE",
    }

    return aliases.get(x, x)


def standardise_metric_file(path, metric):
    df = pd.read_csv(path)

    # Column containing the actual metric value
    value_columns = {
        "RMSE": "rmse",
        "qCRPS": "mean_qcrps",
        "QSS": "qss_0.1",
        "MASE": "mase",
    }

    value_col = value_columns[metric]

    required = {"model", "asset_class", "dataset", value_col}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{metric}: missing columns {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    out = df[
        ["model", "asset_class", "dataset", value_col]
    ].copy()

    out["model"] = out["model"].map(clean_model_name)
    out["asset_class"] = out["asset_class"].astype(str).str.strip()
    out["dataset"] = out["dataset"].map(clean_dataset_name)

    out = out.rename(columns={value_col: "value"})
    out["metric"] = metric

    out["value"] = pd.to_numeric(out["value"], errors="coerce")

    return out


# ============================================================
# LOAD ALL FOUR METRICS
# ============================================================

metric_dfs = {
    metric: standardise_metric_file(path, metric)
    for metric, path in FILES.items()
}

all_metrics = pd.concat(
    metric_dfs.values(),
    ignore_index=True
)

print("Rows by metric:")
print(all_metrics.groupby("metric").size())

print("\nUnique datasets by metric:")
print(all_metrics.groupby("metric")["dataset"].nunique())

print("\nUnique models by metric:")
print(all_metrics.groupby("metric")["model"].nunique())

print("\nAsset classes:")
print(
    all_metrics[["asset_class"]]
    .drop_duplicates()
    .sort_values("asset_class")
    .to_string(index=False)
)


# ============================================================
# OPTIONAL DUPLICATE CHECK
# ============================================================

dupes = all_metrics[
    all_metrics.duplicated(
        subset=["metric", "asset_class", "dataset", "model"],
        keep=False
    )
].sort_values(
    ["metric", "asset_class", "dataset", "model"]
)

if len(dupes):
    print("\nWARNING: duplicate model-dataset rows found:")
    print(dupes.to_string(index=False))
else:
    print("\nNo duplicate model-dataset rows found.")


# ============================================================
# RANKING FUNCTION
# ============================================================

def add_dataset_ranks(df, metric):
    """
    Rank models WITHIN each dataset.

    RMSE, MASE, qCRPS: lower is better.
    QSS: higher is better.

    method='average' handles exact ties fairly.
    """
    out = df.copy()

    higher_is_better = metric == "QSS"

    out["rank"] = out.groupby(
        ["asset_class", "dataset"]
    )["value"].rank(
        ascending=not higher_is_better,
        method="average"
    )

    return out


# ============================================================
# COMPUTE ROBUSTNESS RESULTS FOR ONE METRIC
# ============================================================

def robustness_rankings(df, metric):

    df = df.dropna(subset=["value"]).copy()
    ranked = add_dataset_ranks(df, metric)

    # --------------------------------------------------------
    # 1. Ordinary overall ranking:
    #    each dataset gets equal weight
    # --------------------------------------------------------

    overall = (
        ranked.groupby("model", as_index=False)
        .agg(
            mean_rank=("rank", "mean"),
            n_datasets=("dataset", "nunique"),
        )
        .sort_values("mean_rank")
        .reset_index(drop=True)
    )

    overall["overall_position"] = np.arange(1, len(overall) + 1)
    overall["metric"] = metric


    # --------------------------------------------------------
    # 2. Mean rank within each asset class
    # --------------------------------------------------------

    by_asset = (
        ranked.groupby(
            ["asset_class", "model"],
            as_index=False
        )
        .agg(
            mean_rank=("rank", "mean"),
            n_datasets=("dataset", "nunique")
        )
    )


    # --------------------------------------------------------
    # 3. Equal-asset-class-weighted ranking
    #
    # First average datasets within asset class,
    # THEN average the 8 asset classes.
    # --------------------------------------------------------

    equal_asset = (
        by_asset.groupby("model", as_index=False)
        .agg(
            equal_asset_mean_rank=("mean_rank", "mean"),
            n_asset_classes=("asset_class", "nunique")
        )
        .sort_values("equal_asset_mean_rank")
        .reset_index(drop=True)
    )

    equal_asset["equal_asset_position"] = (
        np.arange(1, len(equal_asset) + 1)
    )

    equal_asset["metric"] = metric


    # --------------------------------------------------------
    # 4. Leave-one-asset-class-out ranking
    # --------------------------------------------------------

    loo_tables = []
    stability_rows = []

    asset_classes = sorted(ranked["asset_class"].unique())

    # Full ranking lookup for correlation
    full_lookup = overall.set_index("model")["mean_rank"]

    for excluded in asset_classes:

        sub = ranked[
            ranked["asset_class"] != excluded
        ].copy()

        loo = (
            sub.groupby("model", as_index=False)
            .agg(
                mean_rank=("rank", "mean"),
                n_datasets=("dataset", "nunique"),
                n_asset_classes=("asset_class", "nunique"),
            )
            .sort_values("mean_rank")
            .reset_index(drop=True)
        )

        loo["position"] = np.arange(1, len(loo) + 1)
        loo["excluded_asset_class"] = excluded
        loo["metric"] = metric

        loo_tables.append(loo)


        # ----------------------------------------------------
        # Spearman correlation with full ranking
        # ----------------------------------------------------

        common_models = sorted(
            set(full_lookup.index)
            & set(loo["model"])
        )

        full_ranks = (
            full_lookup
            .loc[common_models]
            .rank(method="average")
        )

        loo_lookup = loo.set_index("model")["mean_rank"]

        loo_ranks = (
            loo_lookup
            .loc[common_models]
            .rank(method="average")
        )

        rho, p = spearmanr(full_ranks, loo_ranks)

        top_model = loo.iloc[0]["model"]
        top_mean_rank = loo.iloc[0]["mean_rank"]

        stability_rows.append({
            "metric": metric,
            "excluded_asset_class": excluded,
            "spearman_rho_vs_full": rho,
            "spearman_pvalue": p,
            "top_model": top_model,
            "top_mean_rank": top_mean_rank,
            "n_models": len(common_models),
        })


    loo_all = pd.concat(
        loo_tables,
        ignore_index=True
    )

    stability = pd.DataFrame(stability_rows)


    # --------------------------------------------------------
    # Wide leave-one-out table -- easy to read in Excel
    # --------------------------------------------------------

    loo_wide = (
        loo_all.pivot(
            index="model",
            columns="excluded_asset_class",
            values="mean_rank"
        )
    )

    loo_wide.insert(
        0,
        "Full",
        full_lookup
    )

    loo_wide = (
        loo_wide
        .sort_values("Full")
        .reset_index()
    )


    return {
        "ranked_dataset_rows": ranked,
        "overall": overall,
        "by_asset": by_asset,
        "equal_asset": equal_asset,
        "loo_long": loo_all,
        "loo_wide": loo_wide,
        "stability": stability,
    }


# ============================================================
# RUN ALL FOUR METRICS
# ============================================================

results = {}

for metric, df in metric_dfs.items():

    print("\n" + "=" * 70)
    print(metric)
    print("=" * 70)

    res = robustness_rankings(df, metric)
    results[metric] = res

    print("\nOverall ranking:")
    print(
        res["overall"][
            ["overall_position", "model", "mean_rank", "n_datasets"]
        ].to_string(index=False)
    )

    print("\nEqual-asset-class ranking:")
    print(
        res["equal_asset"][
            [
                "equal_asset_position",
                "model",
                "equal_asset_mean_rank",
                "n_asset_classes"
            ]
        ].to_string(index=False)
    )

    print("\nLeave-one-asset-class-out stability:")
    print(
        res["stability"][
            [
                "excluded_asset_class",
                "spearman_rho_vs_full",
                "top_model",
                "top_mean_rank"
            ]
        ].to_string(index=False)
    )


# ============================================================
# SAVE EVERYTHING
# ============================================================

for metric, res in results.items():

    safe_metric = (
        metric.lower()
        .replace("qcrps", "qcrps")
    )

    res["ranked_dataset_rows"].to_csv(
        OUT_DIR / f"{safe_metric}_dataset_ranks.csv",
        index=False
    )

    res["overall"].to_csv(
        OUT_DIR / f"{safe_metric}_overall_ranking.csv",
        index=False
    )

    res["by_asset"].to_csv(
        OUT_DIR / f"{safe_metric}_rank_by_asset_class.csv",
        index=False
    )

    res["equal_asset"].to_csv(
        OUT_DIR / f"{safe_metric}_equal_asset_class_ranking.csv",
        index=False
    )

    res["loo_long"].to_csv(
        OUT_DIR / f"{safe_metric}_leave_one_asset_class_out_long.csv",
        index=False
    )

    res["loo_wide"].to_csv(
        OUT_DIR / f"{safe_metric}_leave_one_asset_class_out_wide.csv",
        index=False
    )

    res["stability"].to_csv(
        OUT_DIR / f"{safe_metric}_ranking_stability.csv",
        index=False
    )

print(f"\nSaved robustness results to:\n{OUT_DIR}")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised\output_weekly"
)

FILES = {
    "RMSE": ROOT / "rmse_summary_all_models.csv",
    "qCRPS": ROOT / "all_models_crps_by_dataset.csv",
    "QSS": ROOT / "qss_summary_all_models.csv",
    "MASE": ROOT / "mase_conventional_summary_all_models.csv",
}

OUT_DIR = ROOT / "robustness_rankings"
OUT_DIR.mkdir(exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def clean_dataset_name(x):
    """
    Standardise dataset names across the metric summary files.

    Examples:
      AUD_USD Historical Data_no_preprocess_lookback-250_horizon-20_slide-1
        -> AUD_USD Historical Data

      AUD_USD Historical Data
        -> AUD_USD Historical Data
    """
    x = str(x).strip()

    suffixes = [
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "_lookback-250_horizon-20_slide-1",
        "_no_preprocess",
    ]

    for suffix in suffixes:
        if x.endswith(suffix):
            x = x[:-len(suffix)]

    return x.strip()


def clean_model_name(x):
    """
    Minimal standardisation only.
    Add aliases here ONLY if inspection shows that the same model
    is written differently across files.
    """
    x = str(x).strip()

    aliases = {
        # Examples -- uncomment/change only if actually needed:
        # "GARCH": "GARCH(1,1)",
        # "Chronos": "Chronos-T5",
        # "Moirai MoE": "Moirai-MoE",
    }

    return aliases.get(x, x)


def standardise_metric_file(path, metric):
    df = pd.read_csv(path)

    # Column containing the actual metric value
    value_columns = {
        "RMSE": "rmse",
        "qCRPS": "mean_qcrps",
        "QSS": "qss_0.1",
        "MASE": "mase",
    }

    value_col = value_columns[metric]

    required = {"model", "asset_class", "dataset", value_col}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{metric}: missing columns {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    out = df[
        ["model", "asset_class", "dataset", value_col]
    ].copy()

    out["model"] = out["model"].map(clean_model_name)
    out["asset_class"] = out["asset_class"].astype(str).str.strip()
    out["dataset"] = out["dataset"].map(clean_dataset_name)

    out = out.rename(columns={value_col: "value"})
    out["metric"] = metric

    out["value"] = pd.to_numeric(out["value"], errors="coerce")

    return out


# ============================================================
# LOAD ALL FOUR METRICS
# ============================================================

metric_dfs = {
    metric: standardise_metric_file(path, metric)
    for metric, path in FILES.items()
}

all_metrics = pd.concat(
    metric_dfs.values(),
    ignore_index=True
)

print("Rows by metric:")
print(all_metrics.groupby("metric").size())

print("\nUnique datasets by metric:")
print(all_metrics.groupby("metric")["dataset"].nunique())

print("\nUnique models by metric:")
print(all_metrics.groupby("metric")["model"].nunique())

print("\nAsset classes:")
print(
    all_metrics[["asset_class"]]
    .drop_duplicates()
    .sort_values("asset_class")
    .to_string(index=False)
)


# ============================================================
# OPTIONAL DUPLICATE CHECK
# ============================================================

dupes = all_metrics[
    all_metrics.duplicated(
        subset=["metric", "asset_class", "dataset", "model"],
        keep=False
    )
].sort_values(
    ["metric", "asset_class", "dataset", "model"]
)

if len(dupes):
    print("\nWARNING: duplicate model-dataset rows found:")
    print(dupes.to_string(index=False))
else:
    print("\nNo duplicate model-dataset rows found.")


# ============================================================
# RANKING FUNCTION
# ============================================================

def add_dataset_ranks(df, metric):
    """
    Rank models WITHIN each dataset.

    RMSE, MASE, qCRPS: lower is better.
    QSS: higher is better.

    method='average' handles exact ties fairly.
    """
    out = df.copy()

    higher_is_better = metric == "QSS"

    out["rank"] = out.groupby(
        ["asset_class", "dataset"]
    )["value"].rank(
        ascending=not higher_is_better,
        method="average"
    )

    return out


# ============================================================
# COMPUTE ROBUSTNESS RESULTS FOR ONE METRIC
# ============================================================

def robustness_rankings(df, metric):

    df = df.dropna(subset=["value"]).copy()
    ranked = add_dataset_ranks(df, metric)

    # --------------------------------------------------------
    # 1. Ordinary overall ranking:
    #    each dataset gets equal weight
    # --------------------------------------------------------

    overall = (
        ranked.groupby("model", as_index=False)
        .agg(
            mean_rank=("rank", "mean"),
            n_datasets=("dataset", "nunique"),
        )
        .sort_values("mean_rank")
        .reset_index(drop=True)
    )

    overall["overall_position"] = np.arange(1, len(overall) + 1)
    overall["metric"] = metric


    # --------------------------------------------------------
    # 2. Mean rank within each asset class
    # --------------------------------------------------------

    by_asset = (
        ranked.groupby(
            ["asset_class", "model"],
            as_index=False
        )
        .agg(
            mean_rank=("rank", "mean"),
            n_datasets=("dataset", "nunique")
        )
    )


    # --------------------------------------------------------
    # 3. Equal-asset-class-weighted ranking
    #
    # First average datasets within asset class,
    # THEN average the 8 asset classes.
    # --------------------------------------------------------

    equal_asset = (
        by_asset.groupby("model", as_index=False)
        .agg(
            equal_asset_mean_rank=("mean_rank", "mean"),
            n_asset_classes=("asset_class", "nunique")
        )
        .sort_values("equal_asset_mean_rank")
        .reset_index(drop=True)
    )

    equal_asset["equal_asset_position"] = (
        np.arange(1, len(equal_asset) + 1)
    )

    equal_asset["metric"] = metric


    # --------------------------------------------------------
    # 4. Leave-one-asset-class-out ranking
    # --------------------------------------------------------

    loo_tables = []
    stability_rows = []

    asset_classes = sorted(ranked["asset_class"].unique())

    # Full ranking lookup for correlation
    full_lookup = overall.set_index("model")["mean_rank"]

    for excluded in asset_classes:

        sub = ranked[
            ranked["asset_class"] != excluded
        ].copy()

        loo = (
            sub.groupby("model", as_index=False)
            .agg(
                mean_rank=("rank", "mean"),
                n_datasets=("dataset", "nunique"),
                n_asset_classes=("asset_class", "nunique"),
            )
            .sort_values("mean_rank")
            .reset_index(drop=True)
        )

        loo["position"] = np.arange(1, len(loo) + 1)
        loo["excluded_asset_class"] = excluded
        loo["metric"] = metric

        loo_tables.append(loo)


        # ----------------------------------------------------
        # Spearman correlation with full ranking
        # ----------------------------------------------------

        common_models = sorted(
            set(full_lookup.index)
            & set(loo["model"])
        )

        full_ranks = (
            full_lookup
            .loc[common_models]
            .rank(method="average")
        )

        loo_lookup = loo.set_index("model")["mean_rank"]

        loo_ranks = (
            loo_lookup
            .loc[common_models]
            .rank(method="average")
        )

        rho, p = spearmanr(full_ranks, loo_ranks)

        top_model = loo.iloc[0]["model"]
        top_mean_rank = loo.iloc[0]["mean_rank"]

        stability_rows.append({
            "metric": metric,
            "excluded_asset_class": excluded,
            "spearman_rho_vs_full": rho,
            "spearman_pvalue": p,
            "top_model": top_model,
            "top_mean_rank": top_mean_rank,
            "n_models": len(common_models),
        })


    loo_all = pd.concat(
        loo_tables,
        ignore_index=True
    )

    stability = pd.DataFrame(stability_rows)


    # --------------------------------------------------------
    # Wide leave-one-out table -- easy to read in Excel
    # --------------------------------------------------------

    loo_wide = (
        loo_all.pivot(
            index="model",
            columns="excluded_asset_class",
            values="mean_rank"
        )
    )

    loo_wide.insert(
        0,
        "Full",
        full_lookup
    )

    loo_wide = (
        loo_wide
        .sort_values("Full")
        .reset_index()
    )


    return {
        "ranked_dataset_rows": ranked,
        "overall": overall,
        "by_asset": by_asset,
        "equal_asset": equal_asset,
        "loo_long": loo_all,
        "loo_wide": loo_wide,
        "stability": stability,
    }


# ============================================================
# RUN ALL FOUR METRICS
# ============================================================

results = {}

for metric, df in metric_dfs.items():

    print("\n" + "=" * 70)
    print(metric)
    print("=" * 70)

    res = robustness_rankings(df, metric)
    results[metric] = res

    print("\nOverall ranking:")
    print(
        res["overall"][
            ["overall_position", "model", "mean_rank", "n_datasets"]
        ].to_string(index=False)
    )

    print("\nEqual-asset-class ranking:")
    print(
        res["equal_asset"][
            [
                "equal_asset_position",
                "model",
                "equal_asset_mean_rank",
                "n_asset_classes"
            ]
        ].to_string(index=False)
    )

    print("\nLeave-one-asset-class-out stability:")
    print(
        res["stability"][
            [
                "excluded_asset_class",
                "spearman_rho_vs_full",
                "top_model",
                "top_mean_rank"
            ]
        ].to_string(index=False)
    )


# ============================================================
# SAVE EVERYTHING
# ============================================================

for metric, res in results.items():

    safe_metric = (
        metric.lower()
        .replace("qcrps", "qcrps")
    )

    res["ranked_dataset_rows"].to_csv(
        OUT_DIR / f"{safe_metric}_dataset_ranks.csv",
        index=False
    )

    res["overall"].to_csv(
        OUT_DIR / f"{safe_metric}_overall_ranking.csv",
        index=False
    )

    res["by_asset"].to_csv(
        OUT_DIR / f"{safe_metric}_rank_by_asset_class.csv",
        index=False
    )

    res["equal_asset"].to_csv(
        OUT_DIR / f"{safe_metric}_equal_asset_class_ranking.csv",
        index=False
    )

    res["loo_long"].to_csv(
        OUT_DIR / f"{safe_metric}_leave_one_asset_class_out_long.csv",
        index=False
    )

    res["loo_wide"].to_csv(
        OUT_DIR / f"{safe_metric}_leave_one_asset_class_out_wide.csv",
        index=False
    )

    res["stability"].to_csv(
        OUT_DIR / f"{safe_metric}_ranking_stability.csv",
        index=False
    )

print(f"\nSaved robustness results to:\n{OUT_DIR}")